# Large Run Step 5: GeoJSON Regions and Corridors

Purpose: run GeoJSON/corridor analyses from existing metric and path tables, with heavy joins submitted as batch work and notebook cells limited to previews and figures.

Outputs: GeoJSON region summaries, path/corridor summary tables, corridor maps, and selected record-section figures.


## Setup
Purpose: load config/output paths and driver settings.

Outputs: printed paths and helper variables.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import textwrap
import time

import pandas as pd

from spatial_vtk.config import SpatialVTKConfig
from spatial_vtk.config.outputs import resolve_output_path
from spatial_vtk.io import load_output_table, preview_output_table, write_output_table


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "spatial_vtk").exists():
            return candidate
    return start


repo_root = find_repo_root()
config_candidates = [
    repo_root / "runs" / "spatial_vtk_config.yaml",
    Path.cwd() / "spatial_vtk_config.yaml",
    Path.cwd() / "runs" / "spatial_vtk_config.yaml",
]
config_path = next((path for path in config_candidates if path.exists()), config_candidates[0])
cfg = SpatialVTKConfig.from_file(config_path).activate()

outputs_root = Path(cfg.path("outputs.root", create_parent=True) or (cfg.root_dir / "runs" / "outputs"))
tables_dir = Path(cfg.path("outputs.tables", create_parent=True) or (outputs_root / "tables"))
figures_dir = Path(cfg.path("outputs.figures", create_parent=True) or (outputs_root / "figures"))
slurm_dir = outputs_root / "slurm"
logs_dir = outputs_root / "logs"
for directory in (tables_dir, figures_dir, slurm_dir, logs_dir):
    directory.mkdir(parents=True, exist_ok=True)

RUN_LOCAL = os.environ.get("SVTK_RUN_LOCAL", "0") == "1"
SUBMIT_SLURM = os.environ.get("SVTK_SUBMIT_SLURM", "0") == "1"
OVERWRITE = os.environ.get("SVTK_OVERWRITE", "0") == "1"
QC_CHUNKSIZE = int(os.environ.get("SVTK_QC_CHUNKSIZE", "1000000"))
PREVIEW_ROWS = int(os.environ.get("SVTK_PREVIEW_ROWS", "5"))

print(f"repo_root: {repo_root}")
print(f"config_path: {config_path}")
print(f"outputs_root: {outputs_root}")
print(f"tables_dir: {tables_dir}")
print(f"figures_dir: {figures_dir}")
print(f"SUBMIT_SLURM={SUBMIT_SLURM} RUN_LOCAL={RUN_LOCAL} OVERWRITE={OVERWRITE}")


## Helpers
Purpose: define skip/status helpers.

Outputs: helper functions only.


In [ ]:
def exists(path):
    return Path(path).exists()


def should_run(*paths):
    return OVERWRITE or not all(Path(path).exists() for path in paths)


def file_info(path):
    path = Path(path)
    if not path.exists():
        return {"path": str(path), "exists": False, "size_gb": None, "modified": None}
    return {
        "path": str(path),
        "exists": True,
        "size_gb": round(path.stat().st_size / 1024**3, 3),
        "modified": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(path.stat().st_mtime)),
    }


def show_files(paths):
    display(pd.DataFrame([file_info(path) for path in paths]))


def submit_script(script_path, *, submit=SUBMIT_SLURM):
    script_path = Path(script_path)
    print(f"script: {script_path}")
    if submit:
        result = subprocess.run(["sbatch", str(script_path)], text=True, capture_output=True, check=False)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        if result.returncode != 0:
            raise RuntimeError(f"sbatch failed with return code {result.returncode}")
    else:
        print(f"Set SVTK_SUBMIT_SLURM=1 and rerun this cell, or submit manually:")
        print(f"sbatch {shlex.quote(str(script_path))}")


def write_python_slurm_script(script_name, python_body, *, job_name, walltime="24:00:00", memory="32G", cpus=1):
    script_path = slurm_dir / script_name
    body = textwrap.dedent(python_body).strip()
    script = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --output={logs_dir}/{job_name}_%j.out
#SBATCH --error={logs_dir}/{job_name}_%j.err
#SBATCH --time={walltime}
#SBATCH --mem={memory}
#SBATCH --cpus-per-task={cpus}

set -euo pipefail
cd {repo_root}
source /project2/jvidale_1700/miniforge3/etc/profile.d/conda.sh || true
conda activate spatial-vtk-py312 || true
python - <<'PYJOB'
{body}
PYJOB
"""
    script_path.write_text(script, encoding="utf-8")
    script_path.chmod(0o755)
    return script_path


def preview_table_path(path, n=PREVIEW_ROWS, columns=None):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    if path.suffix.lower() in {".parquet", ".pq"}:
        import pyarrow.parquet as pq
        parquet = pq.ParquetFile(path)
        available = parquet.schema.names
        selected = [column for column in (columns or available) if column in available]
        for batch in parquet.iter_batches(batch_size=max(int(n), 1), columns=selected):
            return batch.to_pandas().head(n)
        return pd.DataFrame(columns=selected)
    return pd.read_csv(path, nrows=n, usecols=columns)


## Resolve GeoJSON Inputs
Purpose: check that Step 1 and Step 3 outputs needed for region/corridor analysis exist.

Outputs: status table only.


In [ ]:
geojson_path = cfg.path("paths.region_geojson", must_exist=False)
metrics_long_path = resolve_output_path("metrics_long", kind="table", create_parent=True)
path_table_path = resolve_output_path("path_table", kind="table", create_parent=True)
geojson_summaries_path = resolve_output_path("geojson_region_summaries", kind="table", create_parent=True)
comparison_eligible_path = resolve_output_path("comparison_eligible_records", kind="table", create_parent=True)

show_files([geojson_path, metrics_long_path, path_table_path, geojson_summaries_path, comparison_eligible_path])


## Submit Region Summary Build
Purpose: build region summaries in a batch process if they are missing.

Outputs: `geojson_region_summaries` and optional corridor/path summaries.


In [ ]:
if geojson_path is None or not Path(geojson_path).exists():
    print("No region GeoJSON is configured or available.")
elif not metrics_long_path.exists():
    print("metrics_long.parquet is not ready yet; finish Step 3 first.")
elif should_run(geojson_summaries_path):
    script = write_python_slurm_script(
        "step05_geojson_summaries.slurm",
        f"""
        import pandas as pd
        from spatial_vtk.config import SpatialVTKConfig
        from spatial_vtk.config.outputs import resolve_output_path
        from spatial_vtk.io import write_output_table
        from spatial_vtk.spatial.calculate.geojson import annotate_points_with_geojson, classify_paths_with_geojson

        cfg = SpatialVTKConfig.from_file({str(config_path)!r}).activate()
        metrics_path = resolve_output_path("metrics_long", kind="table", cfg=cfg, create_parent=True)
        geojson_path = cfg.path("paths.region_geojson", must_exist=False)
        metrics = pd.read_parquet(metrics_path)
        rows = []
        if geojson_path:
            for point in ("station", "event"):
                try:
                    annotated = annotate_points_with_geojson(metrics, geojson_path, point=point, require_overlap=False)
                    label_col = f"{{point}}_geojson_labels"
                    counts = annotated[label_col].fillna("").replace("", "outside").value_counts().reset_index()
                    counts.columns = ["region", "count"]
                    counts["relation"] = f"{{point}}_inside"
                    rows.append(counts)
                except Exception as exc:
                    rows.append(pd.DataFrame([{{"relation": f"{{point}}_inside", "region": "error", "count": 0, "message": str(exc)}}]))
            try:
                paths = classify_paths_with_geojson(metrics, geojson_path, relation="crosses_boundary", require_overlap=False)
                counts = paths["path_geojson_labels"].fillna("").replace("", "outside").value_counts().reset_index()
                counts.columns = ["region", "count"]
                counts["relation"] = "crosses_boundary"
                rows.append(counts)
            except Exception as exc:
                rows.append(pd.DataFrame([{{"relation": "crosses_boundary", "region": "error", "count": 0, "message": str(exc)}}]))
        summary = pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame(columns=["relation", "region", "count"])
        write_output_table("geojson_region_summaries", summary)
        print(f"geojson_region_summaries rows={{len(summary)}}")
        """,
        job_name="svtk-step05-geojson",
        walltime="08:00:00",
        memory="32G",
        cpus=1,
    )
    submit_script(script)
else:
    print("GeoJSON summary table already exists; skipping.")


## Render Region and Corridor Figures
Purpose: render maps from existing summaries and bounded selected rows only.

Outputs: GeoJSON/corridor figures when `SVTK_MAKE_FIGURES=1`.


In [ ]:
MAKE_FIGURES = os.environ.get("SVTK_MAKE_FIGURES", "0") == "1"
if not MAKE_FIGURES:
    print("Skipping figures. Set SVTK_MAKE_FIGURES=1 to render them.")
else:
    from spatial_vtk.spatial.map.geojson import plot_geojson_polygons_map
    stations = load_output_table("prepared_stations")
    events = load_output_table("prepared_events")
    plot_geojson_polygons_map(geojson_path, stations_df=stations, events_df=events, showfig=False, savefig=True)
    print("Wrote GeoJSON overview figure.")


## Preview Region Outputs
Purpose: inspect bounded samples only.

Outputs: small preview table.


In [ ]:
if geojson_summaries_path.exists():
    display(preview_output_table("geojson_region_summaries", nrows=PREVIEW_ROWS))
else:
    print("GeoJSON summaries are not ready yet.")
